# Adım 3 — Feature Engineering

Bu notebook IEEE-CIS Fraud Detection veri seti üzerinde **kapsamlı feature engineering** gerçekleştirir.  
Adım 1–2'ye bağımlılık yoktur — yalnızca ham CSV dosyaları gereklidir.

**Üretilen Feature Grupları:**

| Grup | Sayı | Açıklama |
|------|------|----------|
| Temporal | 7 | İşlem saati, gün, gece/gündüz, iş günü |
| Entity | 21 | Kart/email/cihaz bazlı istatistikler + velocity |
| Relational | 11 | Tutar oranları, adres çeşitliliği, email uyumu |
| Context | 26 | Rare flag, fraud lift encode, kombinasyon risk |
| **Toplam** | **65** | **434 → 499 kolon** |

**Tasarım Prensipleri:**
- Feature sırası önemlidir: Temporal → Entity → Relational → Context
- `fraud_lift` feature'ları **yalnızca train setinde** üretilir (data leakage önlemi)
- Entity feature'larında `min_tx_count` altı entity için global fallback uygulanır
- Fonksiyonlar işaretleyicidir — veri silmez, yalnızca feature ekler

## 1. Kütüphaneler ve Veri Yükleme

In [1]:
import pandas as pd
import numpy as np

In [2]:
identity_data = pd.read_csv('../train_identity.csv')

trx_data = pd.read_csv('../train_transaction.csv')

## 2. Yardımcı Fonksiyonlar

Adım 1'den alınan `detect_column_types` ve `merge_datasets` fonksiyonları burada yeniden tanımlanıyor.  
Bu notebook bağımsız çalışabilir.

### `detect_column_types(df)`
Kolon başına anlamsal tip belirler: `numeric`, `categorical`, `datetime`.  
Float + yüksek unique → numeric kuralı; isim override katmanı (card, addr, id_, M vb.) uygulanır.

In [3]:
def detect_column_types(df, unique_threshold=20, unique_ratio_threshold=0.01):
    """
    Her kolon için anlamsal tipi belirler.
    pandas dtype + unique sayısı + unique oranını birlikte kullanır.

    unique_threshold: sayısal kolonda bu kadar veya daha az unique varsa kategorik say
    unique_ratio_threshold: unique/satır oranı bunun altındaysa kategorik eğilimli
    """
    rows = []
    n = len(df)

    for col in df.columns:
        series = df[col]
        dtype = series.dtype
        n_unique = series.nunique(dropna=True)
        unique_ratio = n_unique / n if n > 0 else 0

        # İsim override: IEEE-CIS'te int ama aslında kod olan alanlar
        categorical_patterns = ("card", "addr", "ProductCD", "id_",
                                "DeviceType", "DeviceInfo", "P_emaildomain",
                                "R_emaildomain", "M")
        is_name_categorical = col.startswith(categorical_patterns)

        if is_name_categorical:
            semantic_type = "categorical (name-override)"
        elif pd.api.types.is_numeric_dtype(series):
            # Float + bol unique -> neredeyse kesin ölçülen bir nicelik
            if pd.api.types.is_float_dtype(series) and n_unique > unique_threshold:
                semantic_type = "numeric"
            elif n_unique <= unique_threshold or unique_ratio < unique_ratio_threshold:
                semantic_type = "categorical (numeric-coded)"
            else:
                semantic_type = "numeric"
        elif pd.api.types.is_datetime64_any_dtype(series):
            semantic_type = "datetime"
        else:
            semantic_type = "categorical"

        rows.append({
            "column": col,
            "pandas_dtype": str(dtype),
            "n_unique": n_unique,
            "unique_ratio (%)": round(unique_ratio * 100, 2),
            "semantic_type": semantic_type
        })

    return pd.DataFrame(rows)

### `merge_datasets(transaction, identity)`
Left join + satır patlaması kontrolü. Identity eşleşmeyenler NaN alır — bu NaN'ın kendisi fraud sinyali olabilir.

In [4]:
def merge_datasets(transaction, identity, key="TransactionID", how="left"):
    """
    Transaction (ana) ve identity (opsiyonel) tablolarını birleştirir.

    Varsayılan left join: tüm transaction'lar korunur, identity'si
    olmayanlara NaN gelir (bu NaN'in kendisi sinyal olabilir).

    Join öncesi/sonrası satır sayısını ve eşleşme oranını raporlar,
    böylece sessiz satır patlamasını yakalarız.
    """
    n_before = len(transaction)

    # 1. Anahtar her iki tabloda da var mı?
    if key not in transaction.columns:
        raise KeyError(f"'{key}' transaction tablosunda yok.")
    if key not in identity.columns:
        raise KeyError(f"'{key}' identity tablosunda yok.")

    # 2. Anahtar tekilliği kontrolü (patlama riskinin kaynağı)
    tx_dup = transaction[key].duplicated().sum()
    id_dup = identity[key].duplicated().sum()

    # 3. Birleştirme
    merged = transaction.merge(identity, on=key, how=how)
    n_after = len(merged)

    # 4. Eşleşme oranı (identity ile eşleşen transaction sayısı)
    matched = transaction[key].isin(identity[key]).sum()
    match_ratio = matched / n_before * 100 if n_before > 0 else 0

    # 5. Özet rapor
    print("=" * 45)
    print("MERGE RAPORU")
    print("=" * 45)
    print(f"Transaction satır (önce) : {n_before:,}")
    print(f"Identity satır           : {len(identity):,}")
    print(f"Merged satır (sonra)     : {n_after:,}")
    print(f"Toplam kolon             : {merged.shape[1]:,}")
    print("-" * 45)
    print(f"Identity ile eşleşen tx  : {matched:,} ({match_ratio:.2f}%)")
    print(f"Transaction key duplicate: {tx_dup:,}")
    print(f"Identity key duplicate   : {id_dup:,}")

    # 6. Satır patlaması uyarısı
    if n_after > n_before:
        print("\n⚠️  UYARI: Satır sayısı arttı! Identity tarafında")
        print("    tekrar eden key var, join satırları çoğalttı.")
    else:
        print("\n✓ Satır sayısı korundu, patlama yok.")
    print("=" * 45)

    return merged

> **Merge çalıştırılıyor** — 590,540 transaction + 144,233 identity → 590,540 satır (patlama yok):

In [5]:
merged_df = merge_datasets(trx_data, identity_data)

MERGE RAPORU
Transaction satır (önce) : 590,540
Identity satır           : 144,233
Merged satır (sonra)     : 590,540
Toplam kolon             : 434
---------------------------------------------
Identity ile eşleşen tx  : 144,233 (24.42%)
Transaction key duplicate: 0
Identity key duplicate   : 0

✓ Satır sayısı korundu, patlama yok.


In [6]:
types = detect_column_types(merged_df)

## 3. Feature Engineering Fonksiyonları

### `create_temporal_features(df)`

`TransactionDT` (saniye cinsinden offset) üzerinden zaman tabanlı feature'lar üretir.

| Feature | Açıklama |
|---------|----------|
| `tx_hour` | İşlem saati (0–23) |
| `tx_day_of_week` | Haftanın günü (0=Pazartesi) |
| `tx_is_weekend` | Cumartesi/Pazar mı? |
| `tx_is_night` | 22:00–06:00 arası mı? |
| `tx_is_business` | 09:00–17:00, hafta içi mi? |
| `tx_day_part` | morning / afternoon / evening / night |
| `tx_days_since_start` | Veri setinin başından geçen gün sayısı |

> **Fraud bağlantısı:** Gece saatleri ve hafta sonları, azalan gözetim nedeniyle fraud yoğunluğu daha yüksektir.

In [7]:
def create_temporal_features(df: pd.DataFrame, time_col: str = "TransactionDT") -> pd.DataFrame:
    """
    TransactionDT (saniye cinsinden offset) üzerinden
    temporal feature'lar üretir.

    Üretilen feature'lar:
    - tx_hour        : işlem saati (0-23)
    - tx_day_of_week : haftanın günü (0=Pazartesi, 6=Pazar)
    - tx_is_weekend  : Cumartesi veya Pazar mı
    - tx_is_night    : gece mi (00:00-06:00)
    - tx_is_business : mesai saati mi (09:00-17:00, hafta içi)
    - tx_day_part    : morning/afternoon/evening/night (4 dilim)
    - tx_days_since_start : veri setindeki ilk işlemden kaç gün geçmiş

    Tasarım:
    - Orijinal df'i değiştirmez, kopyasına ekler
    - time_col yoksa sessizce boş döner (downstream patlamaz)
    """

    if time_col not in df.columns:
        print(f"[temporal] '{time_col}' kolonu bulunamadı, atlanıyor.")
        return df

    out = df.copy()

    # TransactionDT referans epoch'u: veri setinde bilinen başlangıç noktası yok,
    # minimum değeri 0 kabul edip günlük/saatlik döngüyü çıkarıyoruz.
    seconds = out[time_col]

    out["tx_hour"]         = (seconds // 3600) % 24
    out["tx_day_of_week"]  = (seconds // 86400) % 7
    out["tx_is_weekend"]   = out["tx_day_of_week"].isin([5, 6]).astype(int)
    out["tx_is_night"]     = out["tx_hour"].between(0, 5).astype(int)
    out["tx_is_business"]  = (
        (out["tx_hour"].between(9, 17)) & (out["tx_is_weekend"] == 0)
    ).astype(int)

    # Günün 4 dilimine ayır
    def _day_part(hour):
        if 6 <= hour < 12:
            return "morning"
        elif 12 <= hour < 18:
            return "afternoon"
        elif 18 <= hour < 24:
            return "evening"
        else:
            return "night"

    out["tx_day_part"] = out["tx_hour"].apply(_day_part)

    # Veri setinin başından itibaren kaç gün geçmiş
    out["tx_days_since_start"] = (seconds - seconds.min()) // 86400

    print(f"[temporal] {len([c for c in out.columns if c.startswith('tx_')])} feature üretildi.")
    return out

### `create_entity_features(df)`

Entity bazında (kart, email domain, cihaz) istatistiksel feature'lar üretir ve transaction'lara join eder.

**Her entity için üretilen feature'lar:**
- `{entity}_tx_count` — toplam işlem sayısı
- `{entity}_amt_mean/std/median` — tutar istatistikleri
- `{entity}_amt_zscore` — bu entity'ye göre z-score (anormal tutar tespiti)
- `{entity}_velocity_1h/1d` — son 1 saat / 1 gün işlem hızı

> **Önemli:** `min_tx_count` (varsayılan: 3) altındaki entity'ler için global istatistiklere fallback yapılır.  
> Bu sayede yeni kartlar/kullanıcılar için güvenilmez bireysel istatistik üretilmez.

In [8]:
def create_entity_features(
    df: pd.DataFrame,
    entity_cols: list = None,
    amount_col: str = "TransactionAmt",
    time_col: str = "TransactionDT",
    min_tx_count: int = 3,
) -> pd.DataFrame:
    """
    Entity bazında istatistiksel feature'lar üretir ve
    transaction'lara join eder.

    Her entity için üretilen feature'lar:
    - {entity}_tx_count     : toplam işlem sayısı
    - {entity}_amt_mean     : ortalama işlem tutarı
    - {entity}_amt_std      : işlem tutarı std sapması
    - {entity}_amt_median   : medyan işlem tutarı
    - {entity}_amt_zscore   : bu işlemin entity ortalamasından sapması (kritik sinyal)
    - {entity}_velocity_1h  : son 1 saatteki işlem sayısı (max pencere)
    - {entity}_velocity_1d  : son 1 günlük işlem sayısı (max pencere)

    Tasarım:
    - entity_behavior_report bağımlılığı yok, bağımsız çalışır
    - min_tx_count altındaki entity'ler global ortalama ile doldurulur
    - Orijinal df değişmez
    """

    if entity_cols is None:
        entity_cols = ["card1", "P_emaildomain", "DeviceType"]

    out = df.copy()
    global_amt_mean = out[amount_col].mean()
    global_amt_std  = out[amount_col].std()

    for entity in entity_cols:
        if entity not in out.columns:
            print(f"[entity] '{entity}' bulunamadı, atlanıyor.")
            continue

        prefix = entity.lower().replace("_", "")

        # ── Temel istatistikler ──────────────────────────────────────
        stats = (
            out.groupby(entity)[amount_col]
            .agg(
                tx_count="count",
                amt_mean="mean",
                amt_std="std",
                amt_median="median",
            )
            .reset_index()
        )
        stats.columns = [
            entity,
            f"{prefix}_tx_count",
            f"{prefix}_amt_mean",
            f"{prefix}_amt_std",
            f"{prefix}_amt_median",
        ]

        # min_tx_count altındaki entity'leri işaretle
        low_support = stats["tx_count"] < min_tx_count if "tx_count" in stats.columns else None
        # std NaN olabilir (tek işlem): global std ile doldur
        stats[f"{prefix}_amt_std"] = stats[f"{prefix}_amt_std"].fillna(global_amt_std)

        out = out.merge(stats, on=entity, how="left")

        # ── Z-score (bu işlem entity ortalamasından ne kadar sapıyor) ──
        out[f"{prefix}_amt_zscore"] = (
            (out[amount_col] - out[f"{prefix}_amt_mean"])
            / out[f"{prefix}_amt_std"].replace(0, global_amt_std)
        )

        # ── Velocity ────────────────────────────────────────────────
        if time_col in out.columns:
            out = out.sort_values([entity, time_col])

            for window, label in [(3600, "1h"), (86400, "1d")]:
                col_name = f"{prefix}_velocity_{label}"
                # Her satır için: aynı entity'de son `window` saniyede kaç tx var
                out[col_name] = (
                    out.groupby(entity)[time_col]
                    .transform(
                        lambda t: t.expanding().apply(
                            lambda x: ((x[-1] - x) <= window).sum(),
                            raw=True,
                        )
                    )
                )

        feat_cols = [c for c in out.columns if c.startswith(prefix)]
        print(f"[entity] '{entity}' → {len(feat_cols)} feature üretildi.")

    return out

### `create_relational_features(df)`

Kolonlar arası ilişkilerden türetilen feature'lar. Tek kolonun veremeyeceği bağlamsal bilgiyi yakalar.

**Feature Grupları:**

| Grup | Feature'lar | Açıklama |
|------|-------------|----------|
| Tutar oranları | `rel_amt_per_c1`, `rel_c1_c2_ratio` | C kolonları arası oran |
| D farkları | `rel_d1_d2_diff`, `rel_d1_d3_diff`, `rel_d1_age_ratio` | Zaman aralıkları |
| Kart-adres | `rel_card_addr_count`, `rel_card_addr_diversity` | Kart başına adres çeşitliliği |
| Email | `rel_email_match`, `rel_email_both_known` | Gönderici/alıcı email uyumu |
| Etkileşim | `rel_night_high_amt`, `rel_business_amt` | Gece × yüksek tutar interaksiyonu |

> **Adres çeşitliliği yüksek kart** = birden fazla kimlikle kullanılan kart = fraud sinyali.

In [9]:
def create_relational_features(
    df: pd.DataFrame,
    amount_col: str = "TransactionAmt",
    time_col: str = "TransactionDT",
) -> pd.DataFrame:
    """
    Kolonlar arası ilişkilerden türetilen feature'lar üretir.

    Üretilen feature grupları:
    1. Tutar oranları       : C kolonları arası oran, D farkları
    2. Kart-adres uyumu     : card1+addr1 kombinasyon sıklığı
    3. Email uyumu          : alıcı/gönderici email domain eşleşmesi
    4. D kolonları          : gün bazlı fark ve ivme feature'ları
    5. C kolonları          : kümülatif sayaç sapmaları

    Tasarım:
    - column_relationship_report bağımlılığı yok
    - Kolon yoksa ilgili feature sessizce atlanır
    - Orijinal df değişmez
    """

    out = df.copy()
    produced = []

    # ── 1. Tutar oranları ────────────────────────────────────────────
    # C1: hesaptaki işlem sayacı — tutara oranı anormal artışı yakalar
    if "C1" in out.columns:
        out["rel_amt_per_c1"] = out[amount_col] / (out["C1"].replace(0, np.nan))
        produced.append("rel_amt_per_c1")

    # C1/C2 oranı: iki sayaç birbirine göre dengesizleşirse sinyal
    if all(c in out.columns for c in ["C1", "C2"]):
        out["rel_c1_c2_ratio"] = out["C1"] / (out["C2"].replace(0, np.nan))
        produced.append("rel_c1_c2_ratio")

    # ── 2. D kolonları (gün bazlı geçmişler) ────────────────────────
    # D1: hesabın açılışından bu yana gün sayısı
    # D2: son işlemden bu yana gün sayısı
    # Fark: ne kadar süredir aktif olmayan hesap aniden işlem yapıyor?
    if all(c in out.columns for c in ["D1", "D2"]):
        out["rel_d1_d2_diff"] = out["D1"] - out["D2"]
        produced.append("rel_d1_d2_diff")

    if all(c in out.columns for c in ["D1", "D3"]):
        out["rel_d1_d3_diff"] = out["D1"] - out["D3"]
        produced.append("rel_d1_d3_diff")

    # D1 normalize: işlem günü içinde hesabın yaşına oran
    if "D1" in out.columns and time_col in out.columns:
        days_elapsed = (out[time_col] - out[time_col].min()) / 86400
        out["rel_d1_age_ratio"] = out["D1"] / (days_elapsed.replace(0, np.nan))
        produced.append("rel_d1_age_ratio")

    # ── 3. Kart + adres kombinasyon sıklığı ─────────────────────────
    # Aynı kart farklı adresten işlem yapıyorsa risk artar
    if all(c in out.columns for c in ["card1", "addr1"]):
        combo = out.groupby(["card1", "addr1"]).size().reset_index(name="rel_card_addr_count")
        out = out.merge(combo, on=["card1", "addr1"], how="left")
        produced.append("rel_card_addr_count")

        # Kart başına kaç farklı adres kullanılmış
        addr_diversity = (
            out.groupby("card1")["addr1"]
            .nunique()
            .reset_index(name="rel_card_addr_diversity")
        )
        out = out.merge(addr_diversity, on="card1", how="left")
        produced.append("rel_card_addr_diversity")

    # ── 4. Email domain uyumu ────────────────────────────────────────
    # P (purchaser) ve R (recipient) email domain'i aynı mı?
    if all(c in out.columns for c in ["P_emaildomain", "R_emaildomain"]):
        out["rel_email_match"] = (
            out["P_emaildomain"] == out["R_emaildomain"]
        ).astype(int)
        produced.append("rel_email_match")

        # İkisi de biliniyorsa 1, biri bilinmiyorsa 0
        out["rel_email_both_known"] = (
            out["P_emaildomain"].notna() & out["R_emaildomain"].notna()
        ).astype(int)
        produced.append("rel_email_both_known")

    # ── 5. Tutar × zaman etkileşimi ──────────────────────────────────
    # Gece saatlerinde yüksek tutar: iki sinyalin çarpımı
    if "tx_is_night" in out.columns:
        out["rel_night_high_amt"] = out["tx_is_night"] * out[amount_col]
        produced.append("rel_night_high_amt")

    if "tx_is_business" in out.columns:
        out["rel_business_amt"] = out["tx_is_business"] * out[amount_col]
        produced.append("rel_business_amt")

    print(f"[relational] {len(produced)} feature üretildi: {produced}")
    return out

### `create_context_features(df)`

Bağlamsal ve kombinasyon feature'ları. Kategorik değerlerin istatistiksel bağlamını modele aktarır.

**Feature Grupları:**

1. **Rare categorical flags** — frekansı `rare_threshold` (%1) altında kalan değerler işaretlenir
2. **Frequency encoding** — her kategorik değerin veri setindeki oranı
3. **Fraud lift encoding** — her kategorik değerin fraud oranı / global fraud oranı  
   ⚠️ *Yalnızca train setinde üretilir — test setinde target sızması olur*
4. **ProductCD bazlı context** — ürün tipi × tutar istatistikleri
5. **Kombinasyon risk skoru** — ProductCD × card4 çapraz riski
6. **Global Z-score** — tüm veri setine göre tutar anomalisi

> **Fraud lift > 2** olan kategorik değerler yüksek risk sinyali olarak öne çıkar.

In [10]:
def create_context_features(
    df: pd.DataFrame,
    amount_col: str = "TransactionAmt",
    target_col: str = "isFraud",
    rare_threshold: float = 0.01,
    high_lift_threshold: float = 2.0,
) -> pd.DataFrame:
    """
    Bağlamsal (context) feature'lar üretir.

    Üretilen feature grupları:
    1. Rare categorical flags  : düşük frekanslı kategoriler
    2. ProductCD bazlı context : ürün tipi × tutar etkileşimi
    3. Kart tipi context       : card4/card6 × tutar
    4. Kombinasyon risk skoru  : nadir + fraud_rate yüksek çiftler
    5. Global istatistik sapma : işlemin genel dağılımdan uzaklığı

    Tasarım:
    - rare_combination_report bağımlılığı yok, bağımsız hesaplar
    - target_col varsa fraud_rate encode edilir (train-only, sızıntı riski taşır)
    - rare_threshold: frekansı bu oranın altındaki kategoriler "rare" sayılır
    - high_lift_threshold: fraud_rate / global_rate oranı bu eşiği geçince işaret
    - Orijinal df değişmez
    """

    out = df.copy()
    produced = []
    global_fraud_rate = out[target_col].mean() if target_col in out.columns else None

    # ── 1. Rare categorical flags ────────────────────────────────────
    cat_cols = ["ProductCD", "card4", "card6", "P_emaildomain", "DeviceType"]

    for col in cat_cols:
        if col not in out.columns:
            continue

        freq = out[col].value_counts(normalize=True)
        rare_vals = freq[freq < rare_threshold].index
        feat_name = f"ctx_{col.lower().replace('_','')}_is_rare"
        out[feat_name] = out[col].isin(rare_vals).astype(int)
        produced.append(feat_name)

        # Kategorinin frekansını doğrudan feature olarak ekle
        freq_feat = f"ctx_{col.lower().replace('_','')}_freq"
        out[freq_feat] = out[col].map(freq)
        produced.append(freq_feat)

        # target_col varsa kategori bazında fraud_rate encode et
        # NOT: bunu sadece train setinde kullan, test'e sızdırma
        if target_col in out.columns:
            fraud_rate_map = out.groupby(col)[target_col].mean()
            lift_map = fraud_rate_map / global_fraud_rate
            lift_feat = f"ctx_{col.lower().replace('_','')}_fraud_lift"
            out[lift_feat] = out[col].map(lift_map)
            produced.append(lift_feat)

    # ── 2. ProductCD × tutar etkileşimi ─────────────────────────────
    if "ProductCD" in out.columns:
        prod_amt_mean = out.groupby("ProductCD")[amount_col].mean()
        out["ctx_product_amt_mean"] = out["ProductCD"].map(prod_amt_mean)
        out["ctx_product_amt_diff"] = out[amount_col] - out["ctx_product_amt_mean"]
        out["ctx_product_amt_ratio"] = (
            out[amount_col] / out["ctx_product_amt_mean"].replace(0, np.nan)
        )
        produced += ["ctx_product_amt_mean", "ctx_product_amt_diff", "ctx_product_amt_ratio"]

    # ── 3. Kart tipi × tutar etkileşimi ─────────────────────────────
    if "card4" in out.columns:
        card4_amt_mean = out.groupby("card4")[amount_col].mean()
        out["ctx_card4_amt_mean"] = out["card4"].map(card4_amt_mean)
        out["ctx_card4_amt_diff"] = out[amount_col] - out["ctx_card4_amt_mean"]
        produced += ["ctx_card4_amt_mean", "ctx_card4_amt_diff"]

    # ── 4. Kombinasyon risk skoru ────────────────────────────────────
    # ProductCD + card4 nadir kombinasyonu → risk bayrağı
    if all(c in out.columns for c in ["ProductCD", "card4"]):
        combo_key = out["ProductCD"].astype(str) + "_" + out["card4"].astype(str)
        combo_freq = combo_key.value_counts(normalize=True)
        out["ctx_prod_card4_combo_freq"] = combo_key.map(combo_freq)
        out["ctx_prod_card4_is_rare"] = (
            out["ctx_prod_card4_combo_freq"] < rare_threshold
        ).astype(int)
        produced += ["ctx_prod_card4_combo_freq", "ctx_prod_card4_is_rare"]

        if target_col in out.columns:
            combo_fraud = (
                out.groupby(combo_key)[target_col].mean()
            )
            combo_lift = combo_fraud / global_fraud_rate
            out["ctx_prod_card4_lift"] = combo_key.map(combo_lift)
            # Hem nadir hem lift yüksekse çift bayrak
            out["ctx_prod_card4_high_risk"] = (
                (out["ctx_prod_card4_is_rare"] == 1) &
                (out["ctx_prod_card4_lift"] >= high_lift_threshold)
            ).astype(int)
            produced += ["ctx_prod_card4_lift", "ctx_prod_card4_high_risk"]

    # ── 5. Global istatistik sapması ─────────────────────────────────
    # İşlemin tüm veri setine göre z-skoru
    global_mean = out[amount_col].mean()
    global_std  = out[amount_col].std()
    out["ctx_amt_global_zscore"] = (out[amount_col] - global_mean) / global_std
    out["ctx_amt_log"] = np.log1p(out[amount_col])  # sağa çarpık dağılım için
    produced += ["ctx_amt_global_zscore", "ctx_amt_log"]

    print(f"[context] {len(produced)} feature üretildi.")
    if target_col in out.columns:
        print(f"[context] ⚠️  fraud_lift feature'ları üretildi — sadece train setinde kullan.")

    return out

## 4. Orkestratör — `run_feature_engineering()`

Tüm adımları sırayla çalıştırır. **Sıra kritiktir:**

```
Temporal → Entity → Relational → Context
```

- Relational, temporal flag'lere bağımlıdır (`rel_night_high_amt` için `tx_is_night` gerekir)
- Context, entity istatistiklerine bağımlıdır (fraud lift hesabı için)

**Test seti kullanımı:** `target_col=None` geçilirse fraud lift feature'ları üretilmez.

In [11]:
def run_feature_engineering(
    df: pd.DataFrame,
    entity_cols: list = None,
    amount_col: str = "TransactionAmt",
    time_col: str = "TransactionDT",
    target_col: str = "isFraud",
    rare_threshold: float = 0.01,
    high_lift_threshold: float = 2.0,
    min_tx_count: int = 3,
) -> pd.DataFrame:
    """
    Tüm feature engineering adımlarını sırayla çalıştırır.

    Sıra önemli:
    1. Temporal  — tx_is_night, tx_is_business gibi flag'ler üretilir
    2. Entity    — entity bazlı istatistikler join edilir
    3. Relational— kolon ilişkilerinden feature'lar türetilir
                   (temporal flag'lere bağımlı: rel_night_high_amt vb.)
    4. Context   — bağlamsal ve kombinasyon feature'ları eklenir

    Döndürür:
    - Tüm feature'ların eklendiği DataFrame
    - feature_report: hangi adımda kaç feature üretildi
    """

    print("=" * 50)
    print("Feature Engineering başlıyor...")
    print(f"Başlangıç shape: {df.shape}")
    print("=" * 50)

    initial_cols = set(df.columns)

    # ── Adım 1: Temporal ────────────────────────────────────────────
    print("\n[1/4] Temporal features...")
    df = create_temporal_features(df, time_col=time_col)
    temporal_cols = set(df.columns) - initial_cols

    # ── Adım 2: Entity ──────────────────────────────────────────────
    print("\n[2/4] Entity features...")
    df = create_entity_features(
        df,
        entity_cols=entity_cols,
        amount_col=amount_col,
        time_col=time_col,
        min_tx_count=min_tx_count,
    )
    entity_cols_produced = set(df.columns) - initial_cols - temporal_cols

    # ── Adım 3: Relational ──────────────────────────────────────────
    print("\n[3/4] Relational features...")
    df = create_relational_features(df, amount_col=amount_col, time_col=time_col)
    relational_cols = set(df.columns) - initial_cols - temporal_cols - entity_cols_produced

    # ── Adım 4: Context ─────────────────────────────────────────────
    print("\n[4/4] Context features...")
    df = create_context_features(
        df,
        amount_col=amount_col,
        target_col=target_col,
        rare_threshold=rare_threshold,
        high_lift_threshold=high_lift_threshold,
    )
    context_cols = (
        set(df.columns) - initial_cols - temporal_cols
        - entity_cols_produced - relational_cols
    )

    # ── Özet rapor ──────────────────────────────────────────────────
    total_new = len(df.columns) - len(initial_cols)

    print("\n" + "=" * 50)
    print("Feature Engineering tamamlandı.")
    print(f"  Temporal   : {len(temporal_cols)} feature")
    print(f"  Entity     : {len(entity_cols_produced)} feature")
    print(f"  Relational : {len(relational_cols)} feature")
    print(f"  Context    : {len(context_cols)} feature")
    print(f"  Toplam yeni: {total_new} feature")
    print(f"  Final shape: {df.shape}")
    print("=" * 50)

    return df

## 5. Feature Engineering Çalıştır

Entity kolonları: `card1`, `P_emaildomain`, `DeviceType`  
Beklenen çıktı: 434 → **499 kolon**, 590,540 satır

In [12]:
df_features = run_feature_engineering(
    df=merged_df,
    entity_cols=["card1", "P_emaildomain", "DeviceType"],
    target_col="isFraud",  # test setinde None geç
)

Feature Engineering başlıyor...
Başlangıç shape: (590540, 434)

[1/4] Temporal features...
[temporal] 7 feature üretildi.

[2/4] Entity features...
[entity] 'card1' → 8 feature üretildi.
[entity] 'P_emaildomain' → 7 feature üretildi.
[entity] 'DeviceType' → 7 feature üretildi.

[3/4] Relational features...
[relational] 11 feature üretildi: ['rel_amt_per_c1', 'rel_c1_c2_ratio', 'rel_d1_d2_diff', 'rel_d1_d3_diff', 'rel_d1_age_ratio', 'rel_card_addr_count', 'rel_card_addr_diversity', 'rel_email_match', 'rel_email_both_known', 'rel_night_high_amt', 'rel_business_amt']

[4/4] Context features...
[context] 26 feature üretildi.
[context] ⚠️  fraud_lift feature'ları üretildi — sadece train setinde kullan.

Feature Engineering tamamlandı.
  Temporal   : 7 feature
  Entity     : 21 feature
  Relational : 11 feature
  Context    : 26 feature
  Toplam yeni: 65 feature
  Final shape: (590540, 499)


> **Çıktının ilk satırları:**

In [13]:
df_features.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,ctx_product_amt_diff,ctx_product_amt_ratio,ctx_card4_amt_mean,ctx_card4_amt_diff,ctx_prod_card4_combo_freq,ctx_prod_card4_is_rare,ctx_prod_card4_lift,ctx_prod_card4_high_risk,ctx_amt_global_zscore,ctx_amt_log
0,2987010,0,86549,75.887,C,16496,352.0,117.0,mastercard,134.0,...,33.014647,1.770068,132.387731,-56.500731,0.046429,0,3.202148,0,-0.247280,4.342337
1,2987011,0,86555,16.495,C,4461,375.0,185.0,mastercard,224.0,...,-26.377353,0.384747,132.387731,-115.892731,0.046429,0,3.202148,0,-0.495614,2.861915
2,2987016,0,86620,30.000,H,1790,555.0,150.0,visa,226.0,...,-43.170058,0.410004,133.161806,-103.161806,0.038790,0,1.277579,0,-0.439146,3.433987
3,2987017,0,86668,100.000,H,11492,111.0,150.0,mastercard,219.0,...,26.829942,1.366679,132.387731,-32.387731,0.013442,0,1.486945,0,-0.146458,4.615121
4,2987040,0,87209,75.887,C,13329,569.0,117.0,visa,226.0,...,33.014647,1.770068,133.161806,-57.274806,0.069265,0,3.434805,0,-0.247280,4.342337


## 6. Çıktıyı Kaydet

Parquet formatı seçildi: CSV'ye göre ~5× daha hızlı okuma, ~3× daha az disk alanı, dtype koruması.

> **Çıktı:** `step3_features.parquet` — Adım 4 bu dosyayı input olarak alır.

In [14]:
df_features.to_parquet("step3_features.parquet", index=False)
print(f"Kaydedildi: {df_features.shape}")

Kaydedildi: (590540, 499)


---

## Özet

| Aşama | Feature Sayısı | Öne Çıkan Feature'lar |
|-------|---------------|----------------------|
| Temporal | 7 | `tx_is_night`, `tx_is_weekend`, `tx_hour` |
| Entity | 21 | `card1_amt_zscore`, `card1_velocity_1d`, `card1_velocity_1h` |
| Relational | 11 | `rel_card_addr_diversity`, `rel_email_match`, `rel_night_high_amt` |
| Context | 26 | `ctx_card4_fraud_lift`, `ctx_productcd_fraud_lift`, `ctx_pemaildomain_fraud_lift` |
| **Toplam** | **65** | **434 → 499 kolon** |

**Sonraki adım:** Adım 4 — Multi-Layer Anomali Detection Engine  
Girdi: `step3_features.parquet`